# 04. AI Search 검색 및 RAG 테스트

이미 인덱싱된 법률 데이터를 활용하여 다양한 검색 기법을 실습합니다.

| 검색 기법 | 설명 |
|----------|------|
| **키워드 검색** | BM25 전통적 텍스트 매칭 |
| **벡터 검색** | text-embedding-3-large (3072D) 의미 기반 |
| **하이브리드 검색** | 키워드 + 벡터 RRF 결합 |
| **시맨틱 재랭킹** | Azure AI Search Semantic Ranker (L2R) |
| **필터 + 벡터** | 메타데이터 필터링 후 의미 기반 검색 |
| **Multi-Index 검색** | 4개 인덱스 동시 검색 |
| **RAG** | 검색 결과 + GPT-5.4 종합 답변 |
| **Cross-Index RAG** | 여러 법률 소스 통합 RAG |

권장 순서: `02 → 03 → 04`

⚠️ **실행 위치 필수 조건**  
이 노트북은 **내부망(VNet/Private Endpoint)에 연결된 Remote VM**에서 실행해야 합니다.

## 1. 환경 설정

In [ ]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType, QueryCaptionType
from openai import AzureOpenAI

SEARCH_ENDPOINT  = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT  = os.environ['AZURE_OPENAI_ENDPOINT']
GPT54_DEPLOY     = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')
EMBED_DEPLOY     = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')

from src.search.legal_indexes import LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

credential = DefaultAzureCredential()
mgr = LegalIndexManager(
    endpoint=SEARCH_ENDPOINT,
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

# OpenAI 클라이언트
token_provider = get_bearer_token_provider(
    credential,
    'https://cognitiveservices.azure.com/.default'
)
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version='2025-01-01-preview',
)

print(f"Search Endpoint : {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint : {OPENAI_ENDPOINT}")
print(f"GPT-5.4 배포명  : {GPT54_DEPLOY}")
print()

# 인덱스 통계
stats = mgr.get_all_stats()
print("인덱스 문서 개수:")
for s in stats:
    print(f"  {s['index_name']}: {s['document_count']:,}건")

## 2. 인덱스 스키마 정보 (AI Search Skillset 기반 자동 생성)

### 공통 설계 원칙

| 구분 | 클래스 | 설정 | 용도 |
|------|--------|------|------|
| **Key 필드** | SimpleField | `key=True, filterable=True` | 문서 고유 ID |
| **날짜/번호 메타** | SimpleField | `filterable=True, sortable=True` | 날짜 범위 필터, 정렬 |
| **열거형 메타** | SearchableField | `filterable=True, facetable=True` | 심급·유형 패싯 UI |
| **다중값 메타** | SearchableField | `type=Collection(String), filterable=True` | 관련법령·주제어 정확 필터 |
| **본문 텍스트** | SearchableField | `analyzer_name="ko.microsoft"` | 한국어 키워드 검색 |
| **벡터 임베딩** | SearchField | `searchable=True, hidden=True` | 의미 기반 벡터 검색 (3072D, HNSW) |

**주요 설계 결정:**
- **`Collection(String)` 필터**: "민법,형법" 단일 텍스트 불가 → 배열로 저장하여 람다 필터 가능
- **`hidden=True` 벡터**: 3072 float 응답 페이로드 제외 → 네트워크 비용 감소
- **`analyzer_name="ko.microsoft"`**: 한국어 형태소 분석 (키워드 검색 성능 향상)
- **임베딩 필드 선택**: 전체 본문 아닌 핵심 필드만 임베딩 → 비용 절약

---

### 1. `prec-court-index` — 판례 (대법원·고등법원·지방법원)

| 필드명 | 타입 | Key | Searchable | Filterable | Sortable | Facetable | Hidden | 비고 |
|--------|------|:---:|:-----------:|:----------:|:--------:|:---------:|:------:|------|
| `id` | String | ✅ | | ✅ | | | | 고유 ID |
| `courtName` | String | | ✅ | ✅ | | ✅ | | 법원명 |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | | 사건번호 (텍스트 검색) |
| `judgmentDate` | DateTimeOffset | | | ✅ | ✅ | | | 선고일자 |
| `courtLevel` | String | | ✅ | ✅ | | ✅ | | 심급 (1심/2심/3심) |
| `caseType` | String | | ✅ | ✅ | | ✅ | | 사건종류 (민사/형사/행정) |
| `status` | String | | ✅ | ✅ | | ✅ | | 진행상태 (확정/계류) |
| `relatedLaws` | Collection(String) | | ✅ | ✅ | | | | 관련법령 배열 |
| `keywords` | Collection(String) | | ✅ | ✅ | | | | 주제어 배열 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | | 등록일자 |
| `caseName` | String | | ✅ (ko.microsoft) | | | | | 사건명 |
| `holdings` | String | | ✅ (ko.microsoft) | | | | | 판시사항 |
| `summary` | String | | ✅ (ko.microsoft) | | | | | 판결요지 ← **임베딩 대상** |
| `fullText` | String | | ✅ (ko.microsoft) | | | | | 판결 전문 |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | ✅ | 벡터 3072D (HNSW) |

**Semantic Config**: `prec-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 2. `const-court-index` — 헌법재판소 결정례

| 필드명 | 타입 | Key | Searchable | Filterable | Sortable | Facetable | Hidden | 비고 |
|--------|------|:---:|:-----------:|:----------:|:--------:|:---------:|:------:|------|
| `id` | String | ✅ | | ✅ | | | | 고유 ID |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | | 사건번호 |
| `decisionDate` | DateTimeOffset | | | ✅ | ✅ | | | 결정일자 |
| `decisionType` | String | | ✅ | ✅ | | ✅ | | 결정유형 (합헌/위헌/헌법불합치/각하) |
| `relatedLaws` | Collection(String) | | ✅ | ✅ | | | | 관련법령 배열 |
| `keywords` | Collection(String) | | ✅ | ✅ | | | | 주제어 배열 |
| `fiscalYear` | String | | | ✅ | ✅ | | | 귀속년도 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | | 등록일자 |
| `caseName` | String | | ✅ (ko.microsoft) | | | | | 사건명 |
| `holdings` | String | | ✅ (ko.microsoft) | | | | | 판시사항 |
| `summary` | String | | ✅ (ko.microsoft) | | | | | 결정요지 ← **임베딩 대상** |
| `fullText` | String | | ✅ (ko.microsoft) | | | | | 전문 |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | ✅ | 벡터 3072D (HNSW) |

**Semantic Config**: `const-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 3. `legis-interp-index` — 법제처 해석례

| 필드명 | 타입 | Key | Searchable | Filterable | Sortable | Facetable | Hidden | 비고 |
|--------|------|:---:|:-----------:|:----------:|:--------:|:---------:|:------:|------|
| `id` | String | ✅ | | ✅ | | | | 고유 ID |
| `docNumber` | String | | ✅ | ✅ | ✅ | | | 문서번호 (텍스트 검색) |
| `replyDate` | DateTimeOffset | | | ✅ | ✅ | | | 회신일자 |
| `interpType` | String | | ✅ | ✅ | | ✅ | | 안건종류 (법령해석/법제업무) |
| `relatedLaws` | Collection(String) | | ✅ | ✅ | | | | 관련법령 배열 |
| `keywords` | Collection(String) | | ✅ | ✅ | | | | 주제어 배열 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | | 등록일자 |
| `title` | String | | ✅ (ko.microsoft) | | | | | 안건명 |
| `querySummary` | String | | ✅ (ko.microsoft) | | | | | 질의요지 ← **임베딩 대상** |
| `reply` | String | | ✅ (ko.microsoft) | | | | | 회답 ← **임베딩 대상** |
| `reason` | String | | ✅ (ko.microsoft) | | | | | 이유 |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | ✅ | 벡터 3072D (HNSW) |

**Semantic Config**: `interp-semantic` (title=`title` / content=`querySummary,reply` / keywords=`keywords,relatedLaws`)

---

### 4. `admin-appeal-index` — 행정심판 재결례

| 필드명 | 타입 | Key | Searchable | Filterable | Sortable | Facetable | Hidden | 비고 |
|--------|------|:---:|:-----------:|:----------:|:--------:|:---------:|:------:|------|
| `id` | String | ✅ | | ✅ | | | | 고유 ID |
| `caseNumber` | String | | ✅ | ✅ | ✅ | | | 사건번호 |
| `decisionDate` | DateTimeOffset | | | ✅ | ✅ | | | 재결일자 |
| `decisionType` | String | | ✅ | ✅ | | ✅ | | 재결유형 (인용/기각/각하) |
| `relatedLaws` | Collection(String) | | ✅ | ✅ | | | | 관련법령 배열 |
| `keywords` | Collection(String) | | ✅ | ✅ | | | | 주제어 배열 |
| `committee` | String | | ✅ | ✅ | | ✅ | | 심판위원회 |
| `registrationDate` | DateTimeOffset | | | ✅ | ✅ | | | 등록일자 |
| `caseName` | String | | ✅ (ko.microsoft) | | | | | 사건명 |
| `order` | String | | ✅ (ko.microsoft) | | | | | 주문 ← **임베딩 대상** |
| `reasonSummary` | String | | ✅ (ko.microsoft) | | | | | 이유요약 ← **임베딩 대상** |
| `fullText` | String | | ✅ (ko.microsoft) | | | | | 전문 |
| `summaryEmbedding` | Collection(Single) | | ✅ | | | | ✅ | 벡터 3072D (HNSW) |

**Semantic Config**: `admin-semantic` (title=`caseName` / content=`order,reasonSummary` / keywords=`keywords,relatedLaws`)

---

### Collection(String) OData 필터 예시

```
# 특정 법령 포함
filter="relatedLaws/any(law: law eq '민법')"

# 여러 법령 중 하나
filter="relatedLaws/any(law: law eq '민법' or law eq '형법')"

# 법령 + 날짜 범위 복합
filter="relatedLaws/any(law: law eq '민법') and judgmentDate ge 2020-01-01T00:00:00Z"

# 심급 + 상태 + 날짜
filter="courtLevel eq '3심' and status eq '확정' and judgmentDate ge 2020-01-01T00:00:00Z"
```

## 3. 단일 인덱스 검색 (판례)

### 3-1. 키워드 검색

In [ ]:
query = "개인정보 보호"
results = mgr.search(PREC_INDEX, query=query, top=3, use_semantic=False)

print(f"\n[키워드 검색] '{query}' (판례, 상위 3건)\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"   법원: {r['courtName']}, 심급: {r['courtLevel']}")
    print(f"   점수: {r['score']:.4f}")
    print()

### 3-2. 하이브리드 검색 (키워드 + 벡터)

In [ ]:
query = "법령 개정 절차와 공포 방법은 어떻게 되나요?"

# 쿼리 임베딩 생성
embedding_response = openai_client.embeddings.create(
    input=query,
    model=EMBED_DEPLOY,
)
embedding = embedding_response.data[0].embedding

results = mgr.search(PREC_INDEX, query=query, embedding=embedding, top=3, use_semantic=True)

print(f"\n[하이브리드 + 시맨틱 재랭킹] '{query}' (상위 3건)\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"   점수: {r['score']:.4f}")
    print(f"   판결요지: {r.get('summary', '')[:120]}...")
    print()

## 4. 필터 기반 검색 (메타데이터 필터링)

Collection(String) 필터를 사용한 정밀 검색 예시

In [ ]:
# 대법원 세무 사건만 필터
print("\n[필터 검색] 판례 중 대법원(3심) 세무 사건\n")
results = mgr.search(
    PREC_INDEX,
    query="*",
    filter_expr="courtLevel eq '3심'",
    select=["caseName", "caseNumber", "courtName", "courtLevel"],
    top=5,
)
for r in results[:3]:
    print(f"[{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"  {r['courtName']} - {r['courtLevel']}\n")

## 5. Multi-Index 통합 검색 (4개 인덱스 동시 검색)

4개 법률 인덱스에서 동시에 검색하여 점수 기반으로 통합 정렬합니다.

In [ ]:
def cross_index_search(query: str, top_per_index: int = 3) -> list:
    """4개 인덱스 통합 검색"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=query,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 각 인덱스별 검색
    INDEX_LABELS = {
        PREC_INDEX: "판례",
        CONST_INDEX: "헌재결정례",
        INTERP_INDEX: "법제처해석례",
        ADMIN_INDEX: "행정심판재결례",
    }
    
    all_results = []
    for idx_name in [PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX]:
        results = mgr.search(idx_name, query=query, embedding=embedding, top=top_per_index)
        for r in results:
            r["_index"] = idx_name
            r["_label"] = INDEX_LABELS[idx_name]
        all_results.extend(results)
    
    # 점수 기반 정렬
    all_results.sort(key=lambda x: x.get("score", 0), reverse=True)
    return all_results


# 테스트
query = "개인정보 보호와 관련된 법률적 쟁점"
results = cross_index_search(query, top_per_index=2)

print(f"\n[Multi-Index 검색] '{query}'\n")
print(f"총 {len(results)}건 (4개 인덱스 통합, 상위 8건 표시)\n")

for i, r in enumerate(results[:8], 1):
    title = r.get("caseName") or r.get("title", "N/A")
    print(f"{i}. [{r['_label']}] (score: {r['score']:.4f})")
    print(f"   {title[:80]}")
    print()

## 6. RAG — GPT-5.4 기반 질의응답 (판례)

In [ ]:
def rag_single_index(question: str, index_name: str, top_k: int = 3) -> str:
    """단일 인덱스 RAG"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=question,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 검색
    results = mgr.search(index_name, query=question, embedding=embedding, top=top_k)
    
    # 컨텍스트 구성
    context_parts = []
    for r in results:
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{title}] ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': '당신은 한국 법률 전문가입니다. 제공된 판례 자료를 바탕으로 사용자 질문에 정확하고 체계적으로 답변하세요. 근거 판례번호를 명시하세요.',
            },
            {
                'role': 'user',
                'content': f'## 참고 판례\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_tokens=1000,
        temperature=0.3,
    )
    
    return response.choices[0].message.content


# 테스트
question = "손해배상 청구의 요건은 무엇인가요?"
answer = rag_single_index(question, PREC_INDEX, top_k=3)

print(f"\n[RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)

## 7. Cross-Index RAG — 여러 법률 소스 통합 답변

In [ ]:
def cross_index_rag(question: str, top_per_index: int = 2) -> str:
    """Cross-Index RAG: 4개 인덱스 검색 → GPT-5.4 응답"""
    results = cross_index_search(question, top_per_index=top_per_index)
    
    context_parts = []
    for r in results[:8]:  # 상위 8건
        label = r.get("_label", "기타")
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{label}] {title} ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': (
                    '당신은 한국 법률 전문가입니다. '
                    '판례, 헌법재판소 결정례, 법제처 해석례, 행정심판 재결례 등 '
                    '다양한 법률 자료를 종합하여 사용자 질문에 정확하고 체계적으로 답변하세요.\\n'
                    '근거가 되는 사건번호와 자료 종류를 명시하세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'## 검색된 법률 자료\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_tokens=1500,
        temperature=0.3,
    )
    
    return response.choices[0].message.content


# 테스트
question = "법령 개정 절차에 대해 설명해 주세요"
answer = cross_index_rag(question)

print(f"\n[Cross-Index RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)